# Query Notebook

Goal : To do retrieval, prompt injection and generate answer

In [1]:
import os
import sys

projectRoot = os.path.abspath(os.path.join(os.getcwd(), ".."))
srcPath = os.path.join(projectRoot, "src")
for path_ in (projectRoot, srcPath):   # projectRoot for config.py, srcPath for the pipeline modules
    if path_ not in sys.path:
        sys.path.insert(0, path_)

In [2]:
import rag
import retriever
import prompt as promptModule

print("Done!!")

D:\ProgramFiles\Anaconda\envs\mini-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 867.65it/s] 


Done!!


### Quick try

Runs retriever -> prompt builder -> generator -> sources in one call

In [14]:
queryText = "Why is MoE better than typical tranformer based models?"

In [15]:
result = rag.ask(queryText)

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources:\n{result['sources']}")

Question: Why is MoE better than typical tranformer based models?

Answer:
MoE is better than typical Transformer-based models because it increases capacity without a proportional increase in computation. MoE models use sparse activation, where only a subset of parameters (experts) are activated, leading to lower FLOPs (floating-point operations per second) and better scaling. Experts in MoE models can learn different behaviors, offering specialization. Additionally, MoE models can handle massive capacity, supporting hundreds of billions of parameters, which is not feasible with typical Transformer models that rely heavily on FFN layers that dominate the parameter count. MoE models like Mixtral demonstrate performance comparable to much larger dense models while using significantly fewer active parameters per token.

Sources:
• 24_MixtureOfExperts.md (Chunk 8)
• 24_MixtureOfExperts.md (Chunk 10)
• 24_MixtureOfExperts.md (Chunk 3)
• 24_MixtureOfExperts.md (Chunk 9)
• 24_MixtureOfExperts

### Ask multiple questions in a loop

Handy for quickly comparing answers across several queries

In [10]:
questionList = [
    "What is attention in transformers?",
    "What replaced recurrent architectures like LSTMs?",
]

for question in questionList:
    result = rag.ask(question)
    print(f"Q: {question}")
    print(f"A: {result['answer']}\n")

Q: What is attention in transformers?
A: Attention in transformers allows a model to weigh the importance of different tokens when producing a representation for a given token. This mechanism is based on the attention mechanism and is a key part of the Transformer neural network architecture, which has largely replaced recurrent architectures like LSTMs for most NLP tasks.

Q: What replaced recurrent architectures like LSTMs?
A: Transformers, a neural network architecture based on the attention mechanism.



### Inspect the final prompt sent to the LLM

In [11]:
retrievedChunks = retriever.retrieve(queryText, topK=5)

In [12]:
finalPrompt = promptModule.buildPrompt(queryText, retrievedChunks)
print(finalPrompt)

You are a helpful assistant. Answer the question using ONLY the provided context. If the context does not contain the answer, say you don't know.

Context:
Retrieved Chunk 1:
Therefore:

671B model

may behave like

37B model

in computational cost.

---

# Dense vs MoE Models

| Property            | Dense Model | MoE Model   |
| ------------------- | ----------- | ----------- |
| Parameters Used     | All         | Few         |
| Capacity            | Lower       | Much Higher |
| Compute             | High        | Lower       |
| Communication       | Simpler     | Harder      |
| Training Complexity | Lower       | Higher      |

---

# Applications

---

## Large Language Models

DeepSeek

Mixtral

Gemini

Grok

---

## Multimodal Models

Vision + Language.

---

## Recommendation Systems

Different experts specialize on users.

---

## Speech Models

Specialized processing pathways.

---

# Common Interview Questions

---

## Why Use MoE?

Retrieved Chunk 2:
---

## Why Is Load

### Inspect retrieval only

- Useful for debugging retrieval quality before the LLM is even involved
- We can see exactly which chunks were retrieved and how close they were

In [13]:
retrievedChunks = retriever.retrieve(queryText, topK=5)

for rank, chunk in enumerate(retrievedChunks, start=1):
    print(f"{rank}. {chunk['metadata']['source']} (chunk {chunk['metadata']['chunk_id']}, dist={chunk['distance']:.4f})")
    print(f"   {chunk['text'][:100]}...\n")

1. 24_MixtureOfExperts.md (chunk 8, dist=0.1891)
   Therefore:

671B model

may behave like

37B model

in computational cost.

---

# Dense vs MoE Mode...

2. 24_MixtureOfExperts.md (chunk 10, dist=0.2109)
   ---

## Why Is Load Balancing Needed?

To ensure all experts are utilized.

---

## Why Are MoE Mode...

3. 24_MixtureOfExperts.md (chunk 3, dist=0.2211)
   ↓

Specialist Doctor

Instead of consulting every doctor,

only the relevant experts are involved.

...

4. 24_MixtureOfExperts.md (chunk 9, dist=0.2289)
   ---

## Speech Models

Specialized processing pathways.

---

# Common Interview Questions

---

## ...

5. 24_MixtureOfExperts.md (chunk 5, dist=0.2296)
   When an expert exceeds capacity:

Excess tokens may be dropped or rerouted.

---

# Switch Transform...

